# Domux 真实推理评测（Hack-Astron #4）— transformers 版

**用途**：在免费 T4 上跑通 iFlytekOpenSource/Domux 的真实结构化解析，导出「输入→输出→延迟」证据包。

**⚠️ 为什么不用 vLLM**：Domux 是 Gemma4 多模态架构，只有最新 vLLM 支持，而最新 vLLM 要求 CUDA 13（libcudart.so.13），Colab 只有 CUDA 12，会报 `ImportError: libcudart.so.13`。本版改用 **transformers 直接推理**，完全绕开 vLLM，T4 上稳定可跑。

**操作**：菜单栏 `代码执行程序` → `全部运行`，全程约 10~20 分钟（模型下载占大头，装 transformers 很快）。最后一个 cell 自动下载结果 JSON。

## 1. 安装依赖（transformers + ModelScope，约 2 分钟）

In [ ]:
!pip install -q transformers==5.8.0 accelerate huggingface_hub 2>&1 | tail -3
import transformers, torch, huggingface_hub
print('transformers:', transformers.__version__)
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())


## 2. 从 ModelScope 下载 Domux 模型（约 5GB，已下载会走缓存秒过）

In [ ]:
from huggingface_hub import snapshot_download
MODEL_ID = 'iFlytekOpenSource/Domux'
MODEL_REV = '6c71a32f4d624cadfd9fce9d10240d8068e53456'
MODEL_PATH = snapshot_download(MODEL_ID, revision=MODEL_REV, cache_dir='/content/models')
print('模型已下载到:', MODEL_PATH)
print('文件列表:')
import os
for f in sorted(os.listdir(MODEL_PATH)):
    size = os.path.getsize(os.path.join(MODEL_PATH, f)) / 1024 / 1024
    print(f'  {f:30s} {size:.1f} MB')


## 3. 加载模型到 GPU（T4 / 16G 显存，约 1~2 分钟）

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_PATH)
try:
    model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, torch_dtype=torch.bfloat16, device_map='auto')
except Exception as e:
    print('AutoModelForCausalLM 失败，回退 AutoModel:', type(e).__name__)
    from transformers import AutoModel
    model = AutoModel.from_pretrained(MODEL_PATH, torch_dtype=torch.bfloat16, device_map='auto')
model.eval()
print('模型已加载:', model.__class__.__name__)
print('设备:', model.device, '| dtype:', model.dtype)
n_params = sum(p.numel() for p in model.parameters()) / 1e9
print(f'总参数量: {n_params:.2f}B')

## 4. 冒烟测试：典型智能家居指令 → 七字段槽位

In [ ]:
import time

def generate_text(text):
    try:
        prompt = tok.apply_chat_template([{'role': 'user', 'content': text}], tokenize=False, add_generation_prompt=True)
    except Exception:
        prompt = text
    inputs = tok(prompt, return_tensors='pt').to(model.device)
    pad_id = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=128, do_sample=False, temperature=0, pad_token_id=pad_id)
    return tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

SMOKE = [
    '把客厅的空调调到26度',
    '关闭卧室所有灯',
    '把二楼主卧的窗帘拉开一半',
    '帮我打开厨房的抽油烟机',
    '不要开客厅的灯',
]

for t in SMOKE:
    t0 = time.time()
    out = generate_text(t)
    print(f'输入: {t}')
    print(f'输出: {out}')
    print(f'耗时: {round((time.time()-t0)*1000)} ms')
    print('-'*60)

## 5. 批量评测：格式合规率 + 延迟统计，导出证据 JSON

In [ ]:
import json, time, statistics as st

TEST_SET = None
CANDIDATES = [
    'https://raw.githubusercontent.com/iflytek/domux/main/benchmark/test_set.jsonl',
    'https://raw.githubusercontent.com/iflytek/domux/main/data/test_4057.jsonl',
]
import requests
for url in CANDIDATES:
    try:
        txt = requests.get(url, timeout=15).text
        if txt and not txt.startswith('404'):
            TEST_SET = [json.loads(l)['text'] for l in txt.strip().split('\n') if l.strip()][:200]
            print(f'已加载官方测试集 {len(TEST_SET)} 条: {url}'); break
    except Exception:
        pass

if not TEST_SET:
    TEST_SET = SMOKE * 10
    print(f'官方测试集不可达，回退内置样例 x{len(SMOKE)}，共 {len(TEST_SET)} 条')

SLOT_KEYS = {'action','device','attribute','value','unit','room','floor'}
results, latencies, valid_fmt = [], [], 0

for t in TEST_SET:
    t0 = time.time()
    try:
        out = generate_text(t)
        lat = round((time.time()-t0)*1000)
        latencies.append(lat)
        parsed_ok = False
        try:
            obj = json.loads(out)
            parsed_ok = isinstance(obj, dict) and len(SLOT_KEYS & set(obj.keys())) >= 3
        except Exception:
            parts = out.split('|')
            parsed_ok = len(parts) >= 5
        valid_fmt += int(parsed_ok)
        results.append({'input': t, 'output': out, 'latency_ms': lat, 'format_valid': parsed_ok})
    except Exception as e:
        results.append({'input': t, 'error': str(e)[:200]})

report = {
    'model': f'{MODEL_ID} (HF revision {MODEL_REV})',
    'runtime': f'transformers {transformers.__version__} ({model.__class__.__name__}) on Colab T4',
    'total_cases': len(TEST_SET),
    'format_compliance': round(valid_fmt/len(TEST_SET)*100, 2),
    'latency_avg_ms': round(st.mean(latencies)) if latencies else None,
    'latency_p95_ms': sorted(latencies)[int(len(latencies)*0.95)-1] if latencies else None,
    'timestamp_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'samples': results[:100],
}
with open('domux_eval_result_hf.json', 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)
print(json.dumps({k: report[k] for k in report if k != 'samples'}, ensure_ascii=False, indent=2))
print('已保存 domux_eval_result_hf.json')

## 6. 下载结果文件（自动触发浏览器下载）

In [ ]:
from google.colab import files
files.download('domux_eval_result_hf.json')